# random forest vs xgboost

predicting house price with a random forest and with xgboost. cleaned data comes from `eda.ipynb`.

In [1]:
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
SEED = 42

## 1. load data and hold out one listing

one random listing is taken out before any split. it is not used for training, tuning or testing, and i use it at the end to try the saved model on a house it has never seen.

In [2]:
df = pd.read_csv('../data/processed/houses_clean.csv')
df['is_member'] = df['is_member'].map({True: 1.0, False: 0.0, 'True': 1.0, 'False': 0.0})
print(df.shape)

holdout = df.sample(1, random_state=SEED)
df = df.drop(holdout.index).reset_index(drop=True)
print('rows left for training and testing:', len(df))

features = ['bedrooms', 'bathrooms', 'house_size_sqft', 'land_size_perches', 'posted_month',
            'is_verified', 'location', 'district', 'membership_level', 'is_member']
X = df[features]
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(X_train.shape, X_test.shape)

(27723, 11)
rows left for training and testing: 27722
(22177, 10) (5545, 10)


## 2. preprocessing

all of this sits in one `ColumnTransformer`, so it is fitted on the training rows only.

- numbers: fill missing with the median, `log1p` on house size and land size (both skewed), then standard scaling
- categories (location, district, membership): fill with the most common value and one hot encode. locations with fewer than 30 ads go into one rare group, and a location the model has never seen at prediction time falls into that group too
- target: price is skewed, so the model learns `log(price)` and the result is turned back into rupees

scaling does nothing for trees, but it is cheap and keeps the pipeline general.

In [3]:
log_cols = ['house_size_sqft', 'land_size_perches']
other_num = ['bedrooms', 'bathrooms', 'posted_month', 'is_verified', 'is_member']
cat_cols = ['location', 'district', 'membership_level']

preprocess = ColumnTransformer([
    ('log_num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
        ('scale', StandardScaler()),
    ]), log_cols),
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), other_num),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(min_frequency=30, handle_unknown='infrequent_if_exist', sparse_output=False)),
    ]), cat_cols),
])

def make_pipeline(regressor):
    model = TransformedTargetRegressor(regressor=regressor, func=np.log1p, inverse_func=np.expm1)
    return Pipeline([('prep', preprocess), ('model', model)])

width = preprocess.fit(X_train).get_feature_names_out().shape[0]
print('features after preprocessing:', width)

features after preprocessing: 116


## 3. default random forest and xgboost

both with default parameters, scored on the same 20% test set. rmsle and r2 are on log price, mae is in millions of rupees and the last column is the median error as a percent of the real price.

In [4]:
def evaluate(pipe):
    pred = pipe.predict(X_test)
    log_t, log_p = np.log1p(y_test), np.log1p(pred)
    return {
        'rmsle': mean_squared_error(log_t, log_p) ** 0.5,
        'r2 (log)': r2_score(log_t, log_p),
        'mae (rs m)': mean_absolute_error(y_test, pred) / 1e6,
        'median abs % error': (np.abs(pred - y_test) / y_test).median() * 100,
    }

rf_default = make_pipeline(RandomForestRegressor(random_state=SEED, n_jobs=-1)).fit(X_train, y_train)
xgb_default = make_pipeline(XGBRegressor(random_state=SEED, n_jobs=-1)).fit(X_train, y_train)

models = {'random forest (default)': rf_default, 'xgboost (default)': xgb_default}
pd.DataFrame({name: evaluate(m_) for name, m_ in models.items()}).T.round(3)

,rmsle,r2 (log),mae (rs m),median abs % error
random forest (default),0.392,0.780,7.654,15.43
xgboost (default),0.389,0.783,8.245,18.78


with the reposts removed the score is worse than before (rmsle 0.392 for the forest, it was 0.349), and this is the honest number. xgboost is a little better on rmsle (0.389) but the forest has the lower median error (15.4% against 18.8%). they are very close, so tuning is next.